# Flood-It! — gameplay and ad-reward export

We already finished the retention experiment. This notebook gets the data for the next two questions: can we predict a future recorded ad reward, and can we predict the result of a quickplay attempt from the player's prior gameplay? It does not measure ad acceptance, revenue or the effect of changing difficulty.

Use your existing BigQuery sandbox project. Run each cell. We check the estimated scan first and cap the total at 25 GiB. Do not enable billing. Keep the player-level export out of GitHub.


In [ ]:
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()
project_id = input('Enter your Google Cloud project ID: ').strip()
if not project_id:
    raise ValueError('Enter your own BigQuery project ID.')
client = bigquery.Client(project=project_id)


In [ ]:
queries = {'ad_reward_candidates': "-- Predict whether a new player gets at least one recorded ad reward in days 2-8.\n-- Only the first 24 hours are used for features. No ad offers or revenue are inferred.\nWITH events AS (\n  SELECT user_pseudo_id, event_name, event_timestamp\n  FROM `firebase-public-project.analytics_153293282.events_2*`\n  WHERE user_pseudo_id IS NOT NULL AND TRIM(user_pseudo_id) != ''\n    AND event_timestamp IS NOT NULL\n), first_open AS (\n  SELECT user_pseudo_id, MIN(event_timestamp) AS first_open_us\n  FROM events WHERE event_name = 'first_open' GROUP BY user_pseudo_id\n), data_end AS (\n  SELECT MAX(event_timestamp) AS last_event_us FROM events\n), eligible AS (\n  SELECT f.user_pseudo_id, f.first_open_us,\n    f.first_open_us + 86400000000 AS cutoff_us,\n    f.first_open_us + 691200000000 AS label_end_us\n  FROM first_open AS f CROSS JOIN data_end AS d\n  WHERE f.first_open_us + 691200000000 <= d.last_event_us\n)\nSELECT p.user_pseudo_id, p.first_open_us, p.cutoff_us,\n  COUNTIF(e.event_timestamp < p.cutoff_us) AS early_events,\n  COUNTIF(e.event_timestamp < p.cutoff_us AND e.event_name = 'session_start') AS early_sessions,\n  COUNTIF(e.event_timestamp < p.cutoff_us AND e.event_name = 'level_start') AS early_level_starts,\n  COUNTIF(e.event_timestamp < p.cutoff_us AND e.event_name = 'level_end') AS early_level_ends,\n  COUNTIF(e.event_timestamp < p.cutoff_us AND e.event_name = 'post_score') AS early_scores_posted,\n  COUNTIF(e.event_timestamp < p.cutoff_us AND e.event_name = 'level_start_quickplay') AS early_quickplay_starts,\n  COUNTIF(e.event_timestamp < p.cutoff_us AND e.event_name = 'level_complete_quickplay') AS early_quickplay_wins,\n  COUNTIF(e.event_timestamp < p.cutoff_us AND e.event_name = 'level_fail_quickplay') AS early_quickplay_failures,\n  COUNTIF(e.event_timestamp < p.cutoff_us AND e.event_name = 'level_retry_quickplay') AS early_quickplay_retries,\n  COUNTIF(e.event_timestamp < p.cutoff_us AND e.event_name = 'level_reset_quickplay') AS early_quickplay_resets,\n  COUNTIF(e.event_timestamp < p.cutoff_us AND e.event_name = 'use_extra_steps') AS early_extra_steps,\n  COUNTIF(e.event_timestamp < p.cutoff_us AND e.event_name = 'ad_reward') AS early_ad_rewards,\n  COUNTIF(e.event_timestamp >= p.cutoff_us AND e.event_name = 'ad_reward') AS future_reward_count,\n  COUNTIF(e.event_timestamp >= p.cutoff_us AND e.event_name = 'ad_reward') > 0 AS rewarded_next_7_days\nFROM eligible AS p JOIN events AS e\n  ON p.user_pseudo_id = e.user_pseudo_id\n  AND e.event_timestamp >= p.first_open_us AND e.event_timestamp < p.label_end_us\nGROUP BY p.user_pseudo_id, p.first_open_us, p.cutoff_us\nORDER BY p.first_open_us, p.user_pseudo_id;\n", 'quickplay_candidates': "-- One candidate per recorded quickplay start. Match only unambiguous outcomes.\n-- If a new attempt, retry or reset comes first, do not assume a win or loss.\nWITH raw_events AS (\n  SELECT user_pseudo_id, event_timestamp, event_name,\n    ARRAY(SELECT DISTINCT param.value.string_value FROM UNNEST(event_params) AS param\n      WHERE param.key = 'board' AND param.value.string_value IS NOT NULL\n        AND TRIM(param.value.string_value) != '') AS boards\n  FROM `firebase-public-project.analytics_153293282.events_2*`\n  WHERE user_pseudo_id IS NOT NULL AND TRIM(user_pseudo_id) != ''\n    AND event_timestamp IS NOT NULL\n    AND event_name IN ('level_start_quickplay', 'level_complete_quickplay',\n        'level_fail_quickplay', 'level_retry_quickplay', 'level_reset_quickplay')\n), events AS (\n  SELECT user_pseudo_id, event_timestamp, event_name,\n    IF(ARRAY_LENGTH(boards) = 1, boards[SAFE_OFFSET(0)], NULL) AS board,\n    COUNT(*) OVER (PARTITION BY user_pseudo_id, event_timestamp) AS same_time_events\n  FROM raw_events\n), ordered AS (\n  SELECT *,\n    LEAD(event_timestamp) OVER sequence AS next_timestamp,\n    LEAD(event_name) OVER sequence AS next_name,\n    LEAD(board) OVER sequence AS next_board,\n    LEAD(same_time_events) OVER sequence AS next_same_time_events,\n    COUNTIF(event_name = 'level_start_quickplay') OVER history AS prior_starts,\n    COUNTIF(event_name = 'level_complete_quickplay') OVER history AS prior_wins,\n    COUNTIF(event_name = 'level_fail_quickplay') OVER history AS prior_failures,\n    COUNTIF(event_name = 'level_reset_quickplay') OVER history AS prior_resets,\n    COUNTIF(event_name = 'level_retry_quickplay') OVER history AS prior_retries\n  FROM events\n  WINDOW sequence AS (PARTITION BY user_pseudo_id ORDER BY event_timestamp, event_name, board),\n    history AS (PARTITION BY user_pseudo_id ORDER BY event_timestamp\n      RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)\n), attempts AS (\n  SELECT *, CASE\n    WHEN same_time_events != 1 THEN 'ambiguous_start_time'\n    WHEN board IS NULL THEN 'missing_start_board'\n    WHEN next_timestamp IS NULL THEN 'no_later_event'\n    WHEN next_same_time_events != 1 OR next_timestamp <= event_timestamp THEN 'ambiguous_next_time'\n    WHEN next_timestamp - event_timestamp > 3600000000 THEN 'outside_one_hour'\n    WHEN next_name NOT IN ('level_complete_quickplay', 'level_fail_quickplay') THEN 'interrupted_or_restarted'\n    WHEN next_board IS NULL OR next_board != board THEN 'board_mismatch'\n    WHEN next_name = 'level_complete_quickplay' THEN 'win'\n    ELSE 'loss' END AS match_status\n  FROM ordered WHERE event_name = 'level_start_quickplay'\n)\nSELECT user_pseudo_id, event_timestamp AS start_us,\n  prior_starts, prior_wins, prior_failures, prior_resets, prior_retries,\n  match_status,\n  IF(match_status IN ('win', 'loss'), next_timestamp, NULL) AS outcome_us,\n  CASE WHEN match_status = 'win' THEN 1 WHEN match_status = 'loss' THEN 0 ELSE NULL END AS won_attempt\nFROM attempts\nORDER BY start_us, user_pseudo_id, match_status;\n"}

GIB = 1024 ** 3
TOTAL_LIMIT = 25 * GIB
estimates = {}
for name, sql in queries.items():
    config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=config)
    estimates[name] = job.total_bytes_processed
    if estimates[name] is None:
        raise RuntimeError('BigQuery did not report an estimate.')
    print(f'{name}: {estimates[name] / GIB:.3f} GiB')

print(f'Total estimated scan: {sum(estimates.values()) / GIB:.3f} GiB')
if sum(estimates.values()) > TOTAL_LIMIT:
    raise RuntimeError('The estimated scan exceeds 25 GiB. Stop here.')


In [ ]:
import csv
import json
from datetime import datetime, timezone
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

if input('Check the free quota. Type RUN to export: ').strip() != 'RUN':
    raise RuntimeError('Export cancelled.')

folder = Path(mkdtemp(prefix='flood_it_extended_'))
remaining = TOTAL_LIMIT
manifest = {'exported_utc': datetime.now(timezone.utc).isoformat(), 'queries': []}
for name, sql in queries.items():
    if estimates[name] > remaining or remaining <= 0:
        raise RuntimeError('Remaining query cap is too small.')
    config = bigquery.QueryJobConfig(maximum_bytes_billed=remaining)
    job = client.query(sql, job_config=config)
    rows = job.result()
    columns = [field.name for field in rows.schema]
    count = 0
    with (folder / f'{name}.csv').open('w', newline='', encoding='utf-8') as output:
        writer = csv.writer(output)
        writer.writerow(columns)
        for row in rows:
            writer.writerow([row[column] for column in columns])
            count += 1
    billed = job.total_bytes_billed
    if billed is None:
        raise RuntimeError('BigQuery did not report billed bytes; stop and review.')
    remaining -= max(billed, estimates[name])
    manifest['queries'].append({'name': name, 'rows': count, 'job_id': job.job_id,
                                'estimated_bytes': estimates[name], 'billed_bytes': billed, 'sql': sql})
    print(f'{name}: {count:,} rows saved')

(folder / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = folder / 'flood_it_extended.zip'
with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as bundle:
    for path in sorted(folder.iterdir()):
        if path.suffix in ('.csv', '.json'):
            bundle.write(path, arcname=path.name)
files.download(str(archive))


Upload `flood_it_extended.zip` to our project chat. We will check how many usable observations and positive labels we have before training either model.
